# BatchGeoCache — Colab Runner

This notebook runs the BatchGeoCache pipeline entirely inside the Google Colab VM.

**Data flow:** upload CSV → process/resume packages → inspect results → combine/export → download ZIP → optionally reset local processing files.

> **Privacy:** This notebook does **not** mount Google Drive and does not mount Google Drive. Client files remain in the temporary Colab VM unless you explicitly download or delete them.

> **Before first use:** Set `REPO_URL` below to the public BatchGeoCache GitHub repository URL.


## 1. Install BatchGeoCache

The repository is cloned into `/content`. The package is installed in editable mode so the notebook uses the cloned source directly.

In [ ]:
from pathlib import Path
import os
import sys

# TODO: Replace this placeholder with the actual public GitHub repository URL.
REPO_URL = "https://github.com/<ORG>/BatchGeoCache.git"
REPO_DIR = Path("/content/BatchGeoCache")

if not REPO_DIR.exists():
    !git clone "$REPO_URL" "$REPO_DIR"
else:
    print(f"Repository already exists: {REPO_DIR}")

%cd /content/BatchGeoCache
!pip install -e .

# Make the cloned source immediately importable in the current kernel.
if str(REPO_DIR / "src") not in sys.path:
    sys.path.insert(0, str(REPO_DIR / "src"))

print(f"Using repository: {REPO_DIR}")


## 2. Configure the local Colab workspace

`project_root` is deliberately outside the cloned repository. This keeps client data, package results, logs, state, archives, and final outputs out of the source tree.

In [ ]:
from pathlib import Path

from batchgeocache.config import GeocoderConfig
from batchgeocache.io_utils import ensure_directories

PROJECT_ROOT = Path("/content/BatchGeoCache_runtime")
config = GeocoderConfig(
    project_root=PROJECT_ROOT,
    # Keep these values unless you have a specific reason to change them.
    package_size=100,
)

ensure_directories(config)

RAW_INPUT_DIR = config.data_dir / "00_Inputs(Raw Data)"
RAW_INPUT_DIR.mkdir(parents=True, exist_ok=True)

print(f"Project root: {config.project_root}")
print(f"Package directory: {config.package_dir}")
print(f"State file: {config.state_dir / 'processing_state.json'}")


## 3. Upload the client address CSV

The processor expects these four columns exactly: `Address`, `City`, `Province`, and `Country`.

For a resumable run, keep the same source dataset in this Colab session until processing completes. If you intentionally switch datasets, use the reset cell at the end first.

In [ ]:
from google.colab import files
import io
import pandas as pd

REQUIRED_COLUMNS = {"Address", "City", "Province", "Country"}

print("Select the client's address CSV file to upload.")
uploaded = files.upload()

if not uploaded:
    raise RuntimeError("No file was uploaded.")

if len(uploaded) != 1:
    raise RuntimeError("Please upload exactly one source CSV file.")

source_name, source_bytes = next(iter(uploaded.items()))
source_path = RAW_INPUT_DIR / source_name
source_path.write_bytes(source_bytes)

source_df = pd.read_csv(io.BytesIO(source_bytes))
missing = REQUIRED_COLUMNS - set(source_df.columns)
if missing:
    raise ValueError(
        "Source CSV is missing required columns: " + ", ".join(sorted(missing))
    )

if source_df.empty:
    raise ValueError("Source CSV contains no data rows.")

print(f"Loaded: {source_name}")
print(f"Rows: {len(source_df):,}")
print(f"Columns: {list(source_df.columns)}")
source_df.head()


## 4. Initialize the pipeline and determine resume position

The state file records the last package successfully saved. If a previous package run was interrupted, processing resumes at the next package.

If the saved state belongs to a different dataset, stop and reset rather than continuing with stale state.

In [ ]:
from batchgeocache.geocoder import NominatimGeocoder
from batchgeocache.logging_utils import PipelineLogger
from batchgeocache.package_processor import PackageProcessor
from batchgeocache.state_manager import StateManager

state_manager = StateManager(config)
logger = PipelineLogger(config)
logger.initialize_log()
geocoder = NominatimGeocoder(config)
processor = PackageProcessor(geocoder)

packages = [
    source_df.iloc[start:start + config.package_size].copy()
    for start in range(0, len(source_df), config.package_size)
]
total_packages = len(packages)

state = state_manager.load_state()
if state is not None:
    if state.status != "RUNNING":
        raise RuntimeError(
            f"Unexpected persisted state status: {state.status}. "
            "Use the reset cell before starting again."
        )
    if state.total_packages != total_packages:
        raise RuntimeError(
            "Persisted state belongs to a dataset with a different number of packages "
            f"({state.total_packages} vs {total_packages}). "
            "Use the reset cell before starting a different dataset."
        )
    start_package = state.current_package + 1
    print(
        f"Resume state found: package {state.current_package} of {state.total_packages} "
        f"was last completed. Starting at package {start_package}."
    )
else:
    start_package = 1
    print(f"Starting a new run: {total_packages} package(s).")


## 5. Process packages with checkpointing

Each package is normalized, geocoded, classified into success/partial/failure, written to disk, logged, and then checkpointed. The checkpoint is written **after** the three package CSVs have been saved.

In [ ]:
from batchgeocache.models import PackageSummary
from batchgeocache.io_utils import save_package_outputs

if start_package > total_packages:
    print("All packages are already complete according to the saved state.")
else:
    for package_number in range(start_package, total_packages + 1):
        package_df = packages[package_number - 1]

        metrics, success_df, partial_df, failure_df = processor.process_package(
            package_df=package_df,
            package_number=package_number,
            total_packages=total_packages,
            show_progress=True,
        )

        paths = save_package_outputs(
            config=config,
            package_number=package_number,
            success_df=success_df,
            partial_df=partial_df,
            failure_df=failure_df,
        )

        summary = PackageSummary.from_metrics(
            metrics,
            success_file=paths["success_file"],
            partial_file=paths["partial_file"],
            failure_file=paths["failure_file"],
        )

        logger.log_from_summary(summary)

        state_manager.update_after_package(
            current_package=package_number,
            total_packages=total_packages,
            success_file=summary.success_file,
            partial_file=summary.partial_file,
            failure_file=summary.failure_file,
        )

        print(
            f"Completed package {package_number}/{total_packages}: "
            f"success={summary.success_count}, "
            f"partial={summary.partial_count}, "
            f"failure={summary.failure_count}"
        )

    print("Package processing complete. Checkpoint state is still retained until final export succeeds.")


## 6. Inspect the accumulated results

This is a sanity-check pass over the package CSVs. `boundary_name="calgary"` uses the currently defined Calgary boundary. Pass `None` to skip geographic checking, or change the name when another boundary is added to `boundaries.toml`.

In [ ]:
from batchgeocache.inspect_results import inspect_results

# Change to None for a boundary-neutral run.
BOUNDARY_NAME = "calgary"

inspect_results(config, boundary_name=BOUNDARY_NAME)


## 7. Combine, export, and create the client ZIP

The three package-level result classes are consolidated into three final CSVs and then zipped.

In [ ]:
from batchgeocache.result_combiner import ResultCombiner
from batchgeocache.io_utils import zip_combined_outputs

combiner = ResultCombiner(config)
success_df, partial_df, failure_df = combiner.combine_all()

combined_files = combiner.export_combined(
    success_df=success_df,
    partial_df=partial_df,
    failure_df=failure_df,
)

combined_zip = zip_combined_outputs(config, combined_files)
if combined_zip is None:
    raise RuntimeError("No combined output archive was created.")

print(f"Success rows : {len(success_df):,}")
print(f"Partial rows : {len(partial_df):,}")
print(f"Failure rows : {len(failure_df):,}")
print(f"ZIP created  : {combined_zip}")


## 8. Finalize the completed run

Only after the combined export and ZIP have succeeded do we clear the resume state. The package CSVs are intentionally left in the runtime until you choose whether to reset them.

In [ ]:
state_manager.mark_completed()
print("Processing state cleared. The run is finalized.")


## 9. Download the final ZIP

This uses Colab's direct browser download facility; no Google Drive access is involved.

In [ ]:
from google.colab import files

files.download(str(combined_zip))


## 10. Optional destructive reset

**Opt-in only.** This permanently deletes BatchGeoCache processing files in the local Colab runtime: package outputs, archived runs, combined CSVs/ZIPs, logs, and state. It does not modify the GitHub repository.

Run this only when you are finished with the current local data or deliberately want to start over.

In [ ]:
# DESTRUCTIVE / OPT-IN
# Uncomment and run only when you intentionally want to delete
# all BatchGeoCache-generated local processing files.
#
# from batchgeocache.io_utils import reset_project_files
# reset_project_files(config)


## Session notes

- Colab runtime storage is temporary; a runtime reset/disconnect can remove local processing files.
- For a long run, do not close/restart the runtime until processing and download are complete unless you are relying on the persisted state that remains in the current runtime.
- Do not upload a different dataset into a runtime that contains an active resume state.
- Nominatim requests are rate-limited by `GeocoderConfig`; do not parallelize the notebook's package loop.
